In [2]:
import torch 
import torchvision.datasets as datasets 
import torchvision.transforms as transforms 
import torch.nn as nn 
import matplotlib.pyplot as plt 
from tqdm import tqdm 

In [ ]:
torch.manual_seed(1242)

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307),(0.308))])

mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=16, shuffle=True) # N, C, H ,W 

mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(mnist_testset, batch_size=16, shuffle=True)

device = "cuda" if torch.cuda.is_available() else "cpu"


100%|██████████| 9.91M/9.91M [00:14<00:00, 684kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 89.3kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 861kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.80MB/s]


In [13]:
training_sample = next(iter(train_loader))
print(len(training_sample))
print(training_sample[0].shape)
print(training_sample[1].shape)

2
torch.Size([16, 1, 28, 28])
torch.Size([16])


In [18]:
class MNISTNet(nn.Module): 
    def __init__(self, input_dimensions=28*28, output_classes=10, h_size1=1000, h_size2=2000): 
        super().__init__()
        self.input_dimensions = input_dimensions
        self.linear1 = nn.Linear(input_dimensions, h_size1)
        self.linear2 = nn.Linear(h_size1, h_size2)
        self.linear3 = nn.Linear(h_size2, output_classes)
        self.relu = nn.ReLU()

    def forward(self, x): 
        x = x.view(-1, self.input_dimensions) # the size -1 is inferred from other dimensions
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        return x
    
model = MNISTNet().to(device)

In [19]:
EPOCHS = 1 

def train(train_loader, model, optimizer, loss_fn): 
    model.train()
    total_iterations = 0 

    for epoch in range(EPOCHS): 
        loss_sum = 0 
        num_iterations =0 

        data_iterator = tqdm(train_loader, desc=f"Epoch: {epoch+1}") #train_loader has the entire training 
        #data. we are just wrapping a progress bar on it. 

        for data in data_iterator: #data is a batch of training egs.
            num_iterations+=1 
            total_iterations+=1 

            x, y = data
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad() 
            logits = model(x)
            loss = loss_fn(logits, y)

            loss_sum += loss.item() 
            avg_loss = loss_sum/ num_iterations

            data_iterator.set_postfix(loss=avg_loss) #adding a message in the terminal, displaying the loss

            loss.backward() 
            optimizer.step() 


loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) 
train(train_loader, model, optimizer, loss_fn)

Epoch: 1: 100%|██████████| 3750/3750 [00:15<00:00, 247.46it/s, loss=0.225]


In [20]:
original_weights = {} #keep original weights so that we train a LoRa with it. 
for name, param in model.named_parameters(): 
    original_weights[name] = param.clone().detach()

In [22]:
def test(): 
    correct = 0 
    total =0 
    wrong_counts = [0 for i in range(10)]

    with torch.no_grad():
        for data in tqdm(test_loader, desc='Testing'): 
            x, y = data 
            x = x.to(device)
            y = y.to(device)

            output = model(x.view(-1, 784))
            for idx, i in enumerate(output): 
                if torch.argmax(i) == y[idx]: 
                    correct +=1
                else: wrong_counts[y[idx]] +=1
            
            total+=1 
    
    print(f'Accuracy: {round(correct/total, 3)}')
    for i in range(len(wrong_counts)):
        print(f'wrong counts for the digit {i}: {wrong_counts[i]}')

test()

Testing: 100%|██████████| 625/625 [00:02<00:00, 219.60it/s]

Accuracy: 15.152
wrong counts for the digit 0: 117
wrong counts for the digit 1: 31
wrong counts for the digit 2: 24
wrong counts for the digit 3: 22
wrong counts for the digit 4: 75
wrong counts for the digit 5: 122
wrong counts for the digit 6: 32
wrong counts for the digit 7: 45
wrong counts for the digit 8: 28
wrong counts for the digit 9: 34
